# Module 5: Rate Limiting & API Cost Control with Redis

## Why Rate Limiting Matters for LLM Apps

When you expose an LLM-powered app to users, you face a critical risk:

- **API costs explode** — One user sends 1,000 requests per minute, costing you hundreds of dollars
- **API rate limits** — OpenAI/Groq limits how many requests per minute you can make; one abusive user blocks everyone
- **Denial of Service** — An attacker hammers your endpoint until your LLM budget runs out

**Rate limiting** restricts how many requests a user can make in a given time window.

Redis is perfect for rate limiting because:
1. It's fast (< 1ms) — the rate check adds almost no latency
2. It's atomic — `INCR` is a single atomic operation, no race conditions
3. It handles TTL — counters auto-reset when the time window expires
4. It's shared — works across multiple server instances (unlike in-memory variables)

---

## 1. Fixed Window Rate Limiting

The simplest approach: count requests in a fixed time window.

```
Key: rate:user_42
Value: 3 (current count)
TTL: 57 seconds remaining in window

Limit: 10 requests per 60 seconds

Request comes in:
  INCR rate:user_42  -> 4
  4 <= 10? YES -> allow

...after 7 more requests...

  INCR rate:user_42  -> 11
  11 <= 10? NO -> BLOCK (429 Too Many Requests)

...when TTL hits 0, the key disappears...
  Window resets, user can make 10 more requests
```

The key insight: `INCR` is **atomic** in Redis. Even if 100 concurrent requests hit the same user at the same instant, Redis processes them one at a time — no duplicate increments, no race conditions.

---

In [8]:
import redis
import openai
import time
import os
from dotenv import load_dotenv

load_dotenv()

r = redis.Redis(
    host=os.getenv('REDIS_HOST', 'localhost'),
    port=int(os.getenv('REDIS_PORT', 6379)),
    decode_responses=True
)

client = openai.OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

r.ping()
print("Connected!")

Connected!


In [9]:
class FixedWindowRateLimiter:
    """
    Fixed window rate limiter using Redis INCR.

    Each user gets a counter that resets every `window_seconds`.
    If the counter exceeds `limit`, the request is blocked.
    """

    def __init__(self, redis_client, limit: int = 10, window_seconds: int = 60):
        self.r = redis_client
        self.limit = limit
        self.window = window_seconds

    def check(self, identifier: str) -> dict:
        """
        Check if `identifier` is within rate limit.
        Returns a dict with: allowed, count, remaining, reset_in_seconds
        """
        key = f"rate:fixed:{identifier}"

        # Atomic increment — works safely under concurrent load
        count = self.r.incr(key)

        # On the first request in a window, set the expiry
        if count == 1:
            self.r.expire(key, self.window)

        ttl = self.r.ttl(key)
        remaining = max(0, self.limit - count)
        allowed = count <= self.limit

        if not allowed:
            # Undo the increment — the request was rejected
            self.r.decr(key)

        return {
            "allowed": allowed,
            "count": count if allowed else count - 1,
            "limit": self.limit,
            "remaining": remaining,
            "reset_in_seconds": ttl,
        }

    def reset(self, identifier: str):
        self.r.delete(f"rate:fixed:{identifier}")


# Create a limiter: 5 requests per 30 seconds per user
limiter = FixedWindowRateLimiter(r, limit=5, window_seconds=30)
print("Rate limiter created: 5 requests per 30 seconds")

Rate limiter created: 5 requests per 30 seconds


In [10]:
# Demo: simulate a user making many requests
user_id = "demo_user_42"
limiter.reset(user_id)  # Start fresh

print(f"Simulating 8 requests for user '{user_id}' (limit = 5 per 30s)")
print("-" * 60)
print(f"{'Request':>8}  {'Allowed':>8}  {'Count':>6}  {'Remaining':>10}  {'Reset In':>10}")
print("-" * 60)

for i in range(1, 9):
    result = limiter.check(user_id)
    status = "ALLOWED" if result["allowed"] else "BLOCKED"
    print(f"{i:>8}  {status:>8}  {result['count']:>6}  {result['remaining']:>10}  {result['reset_in_seconds']:>9}s")

limiter.reset(user_id)

Simulating 8 requests for user 'demo_user_42' (limit = 5 per 30s)
------------------------------------------------------------
 Request   Allowed   Count   Remaining    Reset In
------------------------------------------------------------
       1   ALLOWED       1           4         30s
       2   ALLOWED       2           3         30s
       3   ALLOWED       3           2         30s
       4   ALLOWED       4           1         30s
       5   ALLOWED       5           0         30s
       6   BLOCKED       5           0         30s
       7   BLOCKED       5           0         30s
       8   BLOCKED       5           0         30s


## 2. Integrating Rate Limiting with LLM Calls

In a real app, the rate limiter sits between the user's request and the LLM call:

In [11]:
# Production rate limiter: 3 LLM calls per minute per user
llm_limiter = FixedWindowRateLimiter(r, limit=3, window_seconds=60)

def ask_llm_with_rate_limit(user_id: str, prompt: str) -> dict:
    """Ask LLM only if the user is within their rate limit."""
    limit_check = llm_limiter.check(user_id)

    if not limit_check["allowed"]:
        return {
            "success": False,
            "error": "rate_limited",
            "message": f"Rate limit reached. Please wait {limit_check['reset_in_seconds']} seconds.",
            "reset_in_seconds": limit_check["reset_in_seconds"]
        }

    # User is within limit — call the LLM
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=100
    )

    return {
        "success": True,
        "response": response.choices[0].message.content,
        "requests_used": limit_check["count"],
        "requests_remaining": limit_check["remaining"],
        "reset_in_seconds": limit_check["reset_in_seconds"]
    }


# Test: user sends 5 requests (only 3 allowed per minute)
user_id = "test_user_99"
llm_limiter.reset(user_id)

prompts = [
    "What is Redis?",
    "What is caching?",
    "What is an LLM?",
    "What is Python?",   # should be blocked
    "What is Docker?",   # should be blocked
]

for i, prompt in enumerate(prompts, 1):
    result = ask_llm_with_rate_limit(user_id, prompt)
    if result["success"]:
        print(f"Request {i}: ALLOWED  ({result['requests_used']}/{3} used) — {prompt}")
        print(f"           Response: {result['response'][:80]}...")
    else:
        print(f"Request {i}: BLOCKED  — {prompt}")
        print(f"           {result['message']}")
    print()

llm_limiter.reset(user_id)

Request 1: ALLOWED  (1/3 used) — What is Redis?
           Response: Redis, which stands for "REmote DIctionary Server," is an open-source in-memory ...

Request 2: ALLOWED  (2/3 used) — What is caching?
           Response: Caching is a technique used to temporarily store copies of frequently accessed d...

Request 3: ALLOWED  (3/3 used) — What is an LLM?
           Response: An LLM, or Large Language Model, is a type of artificial intelligence model desi...

Request 4: BLOCKED  — What is Python?
           Rate limit reached. Please wait 52 seconds.

Request 5: BLOCKED  — What is Docker?
           Rate limit reached. Please wait 52 seconds.



## 3. Sliding Window Rate Limiting

The fixed window has a flaw: a user can send 5 requests at 11:59:59 and 5 more at 12:00:00 — getting 10 requests in 2 seconds (double the limit).

**Sliding window** solves this by tracking the exact timestamp of every request in the current window using a Redis Sorted Set:

In [12]:
class SlidingWindowRateLimiter:
    """
    Sliding window rate limiter using Redis Sorted Sets.

    Stores the timestamp of each request as a score in a Sorted Set.
    To check the limit: count requests with score > (now - window_seconds).
    Old timestamps are removed automatically.

    More accurate than fixed window but uses slightly more memory.
    """

    def __init__(self, redis_client, limit: int = 10, window_seconds: int = 60):
        self.r = redis_client
        self.limit = limit
        self.window = window_seconds

    def check(self, identifier: str) -> dict:
        key = f"rate:sliding:{identifier}"
        now = time.time()
        window_start = now - self.window

        # Use a pipeline for atomic execution
        pipe = self.r.pipeline()

        # Remove timestamps older than the window
        pipe.zremrangebyscore(key, 0, window_start)

        # Count current requests in the window
        pipe.zcard(key)

        # Add the current request timestamp
        pipe.zadd(key, {str(now): now})

        # Set TTL to clean up the key
        pipe.expire(key, self.window + 1)

        _, current_count, _, _ = pipe.execute()

        # current_count is the count BEFORE this request
        allowed = current_count < self.limit

        if not allowed:
            # Remove the just-added timestamp since we're rejecting
            self.r.zrem(key, str(now))

        return {
            "allowed": allowed,
            "count": current_count + (1 if allowed else 0),
            "limit": self.limit,
            "remaining": max(0, self.limit - current_count - (1 if allowed else 0)),
            "window_seconds": self.window
        }

    def reset(self, identifier: str):
        self.r.delete(f"rate:sliding:{identifier}")


sliding_limiter = SlidingWindowRateLimiter(r, limit=5, window_seconds=60)

test_id = "sliding_demo_user"
sliding_limiter.reset(test_id)

print("Sliding Window Rate Limiter Demo (limit: 5 per 60s)")
print("-" * 55)

for i in range(1, 8):
    result = sliding_limiter.check(test_id)
    status = "ALLOWED" if result["allowed"] else "BLOCKED"
    print(f"Request {i}: {status:8}  Count={result['count']}  Remaining={result['remaining']}")

sliding_limiter.reset(test_id)

Sliding Window Rate Limiter Demo (limit: 5 per 60s)
-------------------------------------------------------
Request 1: ALLOWED   Count=1  Remaining=4
Request 2: ALLOWED   Count=2  Remaining=3
Request 3: ALLOWED   Count=3  Remaining=2
Request 4: ALLOWED   Count=4  Remaining=1
Request 5: ALLOWED   Count=5  Remaining=0
Request 6: BLOCKED   Count=5  Remaining=0
Request 7: BLOCKED   Count=5  Remaining=0
